In [10]:
from __future__ import annotations

import json
import re
from pathlib import Path

import pandas as pd


SOURCE_BASE = Path("../data")
XLSX = SOURCE_BASE / "20251011_additives_all_V2.xlsx"
ADDITIVES_JSON = SOURCE_BASE / "additives.json"

OUT_JSON = Path("./additives_year.json")
DIAG_CSV = Path("./additives_year_match_diagnostics.csv")


def norm(value: object) -> str:
    """Normalize additive names for robust matching."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    return re.sub(r"[^0-9a-z]+", "", str(value).lower())


def split_terms(value: object) -> list[str]:
    """Split additive names or abbreviations."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    text = str(value)
    text = text.replace("，", ",").replace("；", ";").replace("、", ",")
    terms = re.split(r"\s*(?:,|;|\band\b|\+|/)\s*", text)

    return [t.strip() for t in terms if t.strip()]


def parse_year(value: object) -> int | None:
    """Extract the earliest year from a cell."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None

    matches = re.findall(r"(19\d{2}|20\d{2})", str(value))
    if not matches:
        return None

    return min(int(y) for y in matches)


def safe_get(row: pd.Series, col: str) -> object:
    """Safely get a column from a pandas row."""
    return row[col] if col in row.index else None


def main() -> None:
    df = pd.read_excel(XLSX, sheet_name="Sheet1")

    with open(ADDITIVES_JSON, "r", encoding="utf-8") as f:
        additives = json.load(f)

    # =========================
    # 1. Build year-matching database from Excel
    # =========================
    excel_rows: list[dict] = []

    for _, row in df.iterrows():
        year = parse_year(safe_get(row, "years"))
        if year is None:
            continue

        abbr_terms = split_terms(safe_get(row, "additives_abbr"))
        name_terms = split_terms(safe_get(row, "additives"))

        all_text = " ".join(
            [
                str(safe_get(row, "additives_abbr") or ""),
                str(safe_get(row, "additives") or ""),
                str(safe_get(row, "titles") or ""),
            ]
        )

        excel_rows.append(
            {
                "idx": int(safe_get(row, "idx")) if not pd.isna(safe_get(row, "idx")) else -1,
                "year": year,
                "abbr_norm_terms": {norm(x) for x in abbr_terms},
                "name_norm_terms": {norm(x) for x in name_terms},
                "all_norm": norm(all_text),
            }
        )

    # =========================
    # 2. Match additives.json molecules to Excel years
    # =========================
    updated: dict = {}
    diagnostics: list[dict] = []

    for mol_name, meta in additives.items():
        mol_norm = norm(mol_name)

        matches: list[tuple[int, int, str]] = []

        for row in excel_rows:
            reason = ""

            if mol_norm and mol_norm in row["abbr_norm_terms"]:
                reason = "abbr_exact"
            elif mol_norm and mol_norm in row["name_norm_terms"]:
                reason = "name_exact"
            elif mol_norm and mol_norm in row["all_norm"]:
                reason = "text_contains"

            if reason:
                matches.append((row["year"], row["idx"], reason))

        first_year = min((m[0] for m in matches), default=None)

        new_meta = dict(meta)
        new_meta["year"] = first_year
        updated[mol_name] = new_meta

        sorted_matches = sorted(matches, key=lambda x: (x[0], x[1]))

        diagnostics.append(
            {
                "molecule": mol_name,
                "year": first_year,
                "match_count": len(matches),
                "matched_idx": ";".join(str(m[1]) for m in sorted_matches[:20]),
                "match_reason": ";".join(m[2] for m in sorted_matches[:20]),
            }
        )

    # =========================
    # 3. Save output
    # =========================
    with open(OUT_JSON, "w", encoding="utf-8") as f:
        json.dump(updated, f, ensure_ascii=False, indent=2)

    pd.DataFrame(diagnostics).to_csv(DIAG_CSV, index=False, encoding="utf-8-sig")

    matched = sum(1 for item in diagnostics if item["year"] is not None)
    unmatched = [item["molecule"] for item in diagnostics if item["year"] is None]

    print(
        json.dumps(
            {
                "source_excel": str(XLSX),
                "source_json": str(ADDITIVES_JSON),
                "output_json": str(OUT_JSON),
                "diagnostics_csv": str(DIAG_CSV),
                "molecules_total": len(additives),
                "matched_year": matched,
                "unmatched_year": len(unmatched),
                "unmatched_sample": unmatched[:30],
            },
            ensure_ascii=False,
            indent=2,
        )
    )


if __name__ == "__main__":
    main()

{
  "source_excel": "../data/20251011_additives_all_V2.xlsx",
  "source_json": "../data/additives.json",
  "output_json": "additives_year.json",
  "diagnostics_csv": "additives_year_match_diagnostics.csv",
  "molecules_total": 126,
  "matched_year": 126,
  "unmatched_year": 0,
  "unmatched_sample": []
}
